<a href="https://colab.research.google.com/github/javigallego4/TFG/blob/main/MaskRCNN_optimizado_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

from IPython.display import clear_output, display_html
import gc; gc.enable()
import warnings
import os
from pathlib import Path
from tqdm import tqdm

# Basic libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sc
from scipy import stats
import random
import cv2

# Preprocessing libraries
from sklearn.preprocessing import *
import cv2
import albumentations as A

# Library for .tiff files
!pip install tifffile
import tifffile as tiff

# Timm Library
!pip install timm
import timm

# PyTorch 
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau 
from PIL import Image

# MaskRCNN class imports
from typing import Any, Callable, Optional
from torchvision.models.detection.mask_rcnn import _resnet_fpn_extractor
from torch import nn
from torchvision.ops import MultiScaleRoIAlign
from torchvision.ops import misc as misc_nn_ops
from torchvision.transforms._presets import ObjectDetection

from torchvision.models.detection.mask_rcnn import MaskRCNN, MaskRCNNPredictor
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models import *
from torchvision.models.detection.mask_rcnn import _resnet_fpn_extractor

# Deep Lab V3 Backbones
from torchvision.models.segmentation.deeplabv3 import *
from torchvision.models.segmentation import deeplabv3_resnet50, deeplabv3_resnet101, deeplabv3_mobilenet_v3_large, DeepLabV3_ResNet101_Weights

# Metric (mAP)
!pip install torchmetrics
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Weights and biases 
!pip install wandb
import wandb

# Memory usage
import gc
def gc_collect():
    gc.collect()
    torch.cuda.empty_cache()

warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

wandb.login() # 5bf911e7e682da23240c68fb146a222bf0475f7c

clear_output()
print('Number of CPUs: ', os.cpu_count())

DEBUG = False
LOG_IMAGES = False
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

Number of CPUs:  2


# Metric: mean Average Precision (mAP)

In [2]:
if DEBUG: 
  # mAP BBOX
  preds = [
    dict(
      boxes=torch.tensor([[258.0, 41.0, 606.0, 285.0]]),
      scores=torch.tensor([0.536]),
      labels=torch.tensor([0]),
    )
  ]
  target = [
    dict(
      boxes=torch.tensor([[214.0, 41.0, 562.0, 285.0]]),
      labels=torch.tensor([0]),
    )
  ]
  metric = MeanAveragePrecision()
  metric.update(preds, target)
  print(metric.compute())

# Preprocessing Pipeline

In [3]:
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

# Obtenemos todos los nºs de polígonos que nos dan en la siguiente ruta. 

BASE_PATH = "/content/gdrive/MyDrive/juniper/WV3/"
polygon_numbers = os.listdir(BASE_PATH)
polygon_numbers = pd.Series(polygon_numbers).str.split('_', n = 2, expand = True)[1]
polygon_numbers = list(polygon_numbers)
polygon_numbers = sorted(polygon_numbers)

# Guardamos en arrays cada una de las imágenes y máscaras.

p_images, m_images, p_masks, m_masks = [], [], [], []
def load_images(polygon_number):
  # Panchromatic Images
  p_images.append(tiff.imread(BASE_PATH + "polygon_{}/panchromatic.tif".format(polygon_number)))
  p_masks.append(tiff.imread(BASE_PATH + "polygon_{}/mask_panchromatic.tif".format(polygon_number)))
  
  # Multispectral Images
  # Hacemos un permute para poner las imágenes en el formato PyTorch
  m_images.append(tiff.imread(BASE_PATH + "polygon_{}/multispectral.tif".format(polygon_number)))
  m_masks.append(tiff.imread(BASE_PATH + "polygon_{}/mask_multispectral.tif".format(polygon_number)))

def fit_all_images(polygon_numbers):
  with ThreadPoolExecutor(1) as p:
      for i in tqdm(p.map(load_images, polygon_numbers), total=len(polygon_numbers)):
          pass

fit_all_images(polygon_numbers)

if DEBUG: 
  print('\nType: ', type(m_images[0]))
  print('Image Shape', m_images[0].shape)
  print('Mask Shape', m_masks[0].shape)

  fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(8,4))
  axes[0].imshow(m_images[1][:,:,:3])
  axes[1].imshow(m_masks[1])

  resized_image = cv2.resize(m_images[1], (215, 256), interpolation = cv2.INTER_NEAREST)
  resized_mask = cv2.resize(m_masks[1], (215, 256), interpolation = cv2.INTER_NEAREST)

  fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(8,4))
  axes[0].imshow(resized_image[:,:,:3])
  axes[1].imshow(resized_mask)

  resized_image = cv2.resize(resized_image, (21, 25), interpolation = cv2.INTER_NEAREST)
  resized_mask = cv2.resize(resized_mask, (21, 25), interpolation = cv2.INTER_NEAREST)

  fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(8,4))
  axes[0].imshow(resized_image[:,:,:3])
  axes[1].imshow(resized_mask)

  # Transpose the image and mask
  image = m_images[1].transpose(1,0,2)
  mask = m_masks[1].transpose(1,0)

  # Flip the image and mask vertically
  image = cv2.flip(image, 1)
  mask = cv2.flip(mask, 1)

100%|██████████| 261/261 [00:02<00:00, 114.70it/s]


## Pansharpening

In [4]:
def inverse_pca_image(sharpened_img, pca):
  """
    Performs inverse PCA transformation to the given image.

    Parameters:
    - sharpened_img: image to be transformed
    - pca: PCA object with loadings from previous fit_transform()
  """

  X = sharpened_img.permute(1,2,0).numpy()

  b0 = X[:,:,0]
  b1 = X[:,:,1]
  b2 = X[:,:,2]
  b3 = X[:,:,3]
  b4 = X[:,:,4]
  b5 = X[:,:,5]
  b6 = X[:,:,6]
  b7 = X[:,:,7]

  pca_df = pd.DataFrame({'B0': b0.flatten(), 'B1': b1.flatten(), 'B2': b2.flatten(), 'B3':b3.flatten(), 
              'B4': b4.flatten(), 'B5': b5.flatten(), 'B6': b6.flatten(), 'B7':b7.flatten()})

  img = pca.inverse_transform(pca_df)

  for i in range(8): 
    sharpened_img[i, :, :] = torch.from_numpy(img[:,i].reshape((sharpened_img.shape[1], sharpened_img.shape[2])))

  return sharpened_img

def histogram_match(pan, band):
    """
    Performs histogram matching between the panchromatic image and the multispectral band given.

    Parameters:
    - pan: torch tensor of shape (height, width)
    - band: torch tensor of shape (height, width)

    Returns:
    - matched_panchromatic: histogram matched PAN imagery
    """

    # Fórmula UGR: https://www.google.com/url?sa=t&rct=j&q=&esrc=s&source=web&cd=&cad=rja&uact=8&ved=2ahUKEwicxYCXrYv9AhXRhqQKHYoUDFQQFnoECAkQAQ&url=https%3A%2F%2Fccia.ugr.es%2Fvip%2Ffiles%2Fbooks%2Fpaper_amro_mateos.pdf&usg=AOvVaw3wn01QiErGCJLtNZUg-oKe
    matched_panchromatic = (pan - pan.mean()) * (band.std() / pan.std()) + band.mean()

    return matched_panchromatic

def pansharpen_image(multispectral_image, panchromatic_image, method):
    """
    Pansharpens the given MS image based on different techniques.

    Parameters:
    - multispectral_image: torch tensor of shape (channels, height, width)
    - panchromatic_image: torch tensor of shape (height, width)
    - method: type of pansharpening technique

    Returns:
    - sharpened_img: torch tensor with same shape as input multispectral image
    """
    
    if method == 'PCA': 
      # Forward Transform: perform PCA and replace 1st component with panchromatic band
      sharpened_img, pca = pca_image(multispectral_image, 8, True)
      sharpened_img = torchvision.transforms.Resize((panchromatic_image.shape[0], panchromatic_image.shape[1]))(sharpened_img)

      # Histogram Matching: the PAN Imagery is matched with the PC1 band
      matched_panchromatic = histogram_match(panchromatic_image, sharpened_img[0, :, :])

      # Component Substitution
      sharpened_img[0, :, :] = matched_panchromatic

      # Reverse Transform: undo PCA
      sharpened_img = inverse_pca_image(sharpened_img, pca)
      
    if method == 'Simple Mean':
      multispectral_image = torchvision.transforms.Resize((panchromatic_image.shape[0], panchromatic_image.shape[1]))(multispectral_image)
      sharpened_img = torch.randn(multispectral_image.shape[0], multispectral_image.shape[1], multispectral_image.shape[2])
      for i in range(multispectral_image.shape[0]):
        # Histogram Matching for each band
        matched_panchromatic = histogram_match(panchromatic_image, multispectral_image[i, :, :])        
        sharpened_img[i, :, :] = 0.5 * (multispectral_image[i, :, :] + matched_panchromatic)

    if method == 'Brovey': 
        multispectral_image = torchvision.transforms.Resize((panchromatic_image.shape[0], panchromatic_image.shape[1]))(multispectral_image)
        sharpened_img = torch.randn(multispectral_image.shape[0], multispectral_image.shape[1], multispectral_image.shape[2])

        M = 0
        for i in range(multispectral_image.shape[0]):
          M += multispectral_image[i, :, :]
        M /= multispectral_image.shape[0]

        for i in range(multispectral_image.shape[0]):
          # Histogram Matching for each band
          matched_panchromatic = histogram_match(panchromatic_image, multispectral_image[i, :, :])
          sharpened_img[i, :, :] = (matched_panchromatic / M) * multispectral_image[i, :, :]

    if method == 'HSV': 
      multispectral_image = torchvision.transforms.Resize((panchromatic_image.shape[0], panchromatic_image.shape[1]))(multispectral_image)
      # Forward Transform
      red = multispectral_image[4, :, :]
      green = multispectral_image[2, :, :]
      blue = multispectral_image[1, :, :]

      I = 0.577 * (red+green+blue)
      v1 = -0.408 * (red+green) + 0.816 * blue
      v2 = -0.707 * (red+green) + 1.703 * blue

      H = torch.atan(v2/v1)
      S = torch.sqrt(torch.pow(v1,2) + torch.pow(v2,2))

      # Histogram Matching
      matched_panchromatic = histogram_match(panchromatic_image, I)

      # Reverse Transformation
      new_red = 0.577 * matched_panchromatic - 0.408 * v1 - 0.707 * v2
      new_green = 0.577 * matched_panchromatic - 0.408 * v1 - 0.816 * v2
      new_blue = 0.577 * matched_panchromatic - 0.816 * v1

      sharpened_img = multispectral_image
      sharpened_img[4, :, :] = new_red
      sharpened_img[2, :, :] = new_green
      sharpened_img[1, :, :] = new_blue

    if method == 'IHS1':
      multispectral_image = torchvision.transforms.Resize((panchromatic_image.shape[0], panchromatic_image.shape[1]))(multispectral_image)
      # Forward Transform
      red = multispectral_image[4, :, :]
      green = multispectral_image[2, :, :]
      blue = multispectral_image[1, :, :]

      I = 1/np.sqrt(3) * (red+green+blue)
      v1 = -1/np.sqrt(6) * (red+green) + 2/np.sqrt(6) * blue
      v2 = 1/np.sqrt(2) * (-red+green)

      H = torch.atan(v2/v1)
      S = torch.sqrt(torch.pow(v1,2) + torch.pow(v2,2))

      # Histogram Matching
      matched_panchromatic = histogram_match(panchromatic_image, I)

      # Reverse Transformation
      v1 = S * torch.cos(H)
      v2 = S * torch.sin(H)

      new_red = 1/np.sqrt(3) * matched_panchromatic -1/np.sqrt(6) * v1 - 1/np.sqrt(2) * v2
      new_green = 1/np.sqrt(3) * matched_panchromatic -1/np.sqrt(6) * v1 + 1/np.sqrt(2) * v2
      new_blue = 1/np.sqrt(3) * matched_panchromatic + 2/np.sqrt(6) * v1

      sharpened_img = multispectral_image
      sharpened_img[4, :, :] = new_red
      sharpened_img[2, :, :] = new_green
      sharpened_img[1, :, :] = new_blue

    if method == 'IHS2':
      multispectral_image = torchvision.transforms.Resize((panchromatic_image.shape[0], panchromatic_image.shape[1]))(multispectral_image)
      # Forward Transform
      red = multispectral_image[4, :, :]
      green = multispectral_image[2, :, :]
      blue = multispectral_image[1, :, :]

      I = 1/3 * (red+green+blue)
      v1 = -1/np.sqrt(6) * (red+green) + 2/np.sqrt(6) * blue
      v2 = 1/np.sqrt(6) * red - 2 /np.sqrt(6) * green

      H = torch.atan(v2/v1)
      S = torch.sqrt(torch.pow(v1,2) + torch.pow(v2,2))

      # Histogram Matching
      matched_panchromatic = histogram_match(panchromatic_image, I)

      # Reverse Transformation
      v1 = S * torch.cos(2*np.pi*H)
      v2 = S * torch.sin(2*np.pi*H)

      new_red = 1 * matched_panchromatic -0.204124 * v1 - 0.612372 * v2
      new_green = 1 * matched_panchromatic -0.204124 * v1 + 0.612372 * v2
      new_blue = 1 * matched_panchromatic + 0.408248 * v1

      sharpened_img = multispectral_image
      sharpened_img[4, :, :] = new_red
      sharpened_img[2, :, :] = new_green
      sharpened_img[1, :, :] = new_blue

    if method == 'IHS3':
      multispectral_image = torchvision.transforms.Resize((panchromatic_image.shape[0], panchromatic_image.shape[1]))(multispectral_image)
      # Forward Transform
      red = multispectral_image[4, :, :]
      green = multispectral_image[2, :, :]
      blue = multispectral_image[1, :, :]

      I = 1/3 * (red+green+blue)
      v1 = -1/np.sqrt(6) * (red+green) + 2/np.sqrt(6) * blue
      v2 = 1/np.sqrt(6) * red - 1/np.sqrt(6) * green

      H = torch.atan(v2/v1)
      S = torch.sqrt(torch.pow(v1,2) + torch.pow(v2,2))

      # Histogram Matching
      matched_panchromatic = histogram_match(panchromatic_image, I)

      # Reverse Transformation
      new_red = 1 * matched_panchromatic -1/np.sqrt(6) * v1 +3/np.sqrt(6) * v2
      new_green = 1 * matched_panchromatic -1/np.sqrt(6) * v1 -3/np.sqrt(6) * v2
      new_blue = 1 * matched_panchromatic + 2/np.sqrt(6) * v1

      sharpened_img = multispectral_image
      sharpened_img[4, :, :] = new_red
      sharpened_img[2, :, :] = new_green
      sharpened_img[1, :, :] = new_blue

    #if method == 'Wavelet': 

    #if method == ''

    return sharpened_img

def pansharpening(method):
  """
  Performs pansharpening to every image in the dataset.
  """

  for i in range(len(p_images)):
    img = pansharpen_image(torch.from_numpy(m_images[i]).permute(2,0,1), torch.from_numpy(p_images[i]), method)
    m_images[i] = img.permute(1,2,0).numpy()

if DEBUG: 
  #scaling()
  for i in range(2):

    # Vemos como sería la imagen multispectral con la resolucion de la pancromática.
    multispectral_image = torchvision.transforms.Resize((p_images[i].shape[0], p_images[i].shape[1]))(torch.from_numpy(m_images[i].astype(np.int)).permute(2,0,1))
    plt.imshow(multispectral_image[:3, :, :].permute(1,2,0))
    plt.show()

    # Aplicamos el pansharpening para observar el cambio.
    img = pansharpen_image(torch.from_numpy(m_images[i].astype(np.float)).permute(2,0,1), torch.from_numpy(p_images[i].astype(np.float)), 'IHS3')
    fig, axes = plt.subplots(nrows = 2, ncols = 2, figsize = (12, 8))

    axes[0,0].imshow(img[:3, :, :].permute(1,2,0))
    axes[0,1].imshow(img[:3, :, :].permute(1,2,0))
    axes[0,1].imshow(p_masks[i], alpha = .5)

    axes[1,0].imshow(p_images[i], cmap = 'gray')
    axes[1,1].imshow(p_images[i])
    axes[1,1].imshow(p_masks[i], alpha = .5)
    plt.show()

## Scaling

In [5]:
def scale_panchromatic_image(image, transformer = MinMaxScaler()):
  '''Returns input panchromatic image with its values being scaled to the [0,1] interval. '''

  img = transformer.fit_transform(image, )
  return img

def scale_multispectral_image(image, bands = 8, transformer = MinMaxScaler()):
  '''Returns input multispectral image with its values being scaled to the [0,1] interval. '''

  b0 = image[:,:,0]
  b1 = image[:,:,1]
  b2 = image[:,:,2]
  if bands == 8: 
    b3 = image[:,:,3]
    b4 = image[:,:,4]
    b5 = image[:,:,5]
    b6 = image[:,:,6]
    b7 = image[:,:,7]

  # As before, we perform some scaling first
  sc = transformer
  b0 = sc.fit_transform(b0)
  b1 = sc.fit_transform(b1)
  b2 = sc.fit_transform(b2)
  if bands == 8: 
    b3 = sc.fit_transform(b3)
    b4 = sc.fit_transform(b4)
    b5 = sc.fit_transform(b5)
    b6 = sc.fit_transform(b6)
    b7 = sc.fit_transform(b7)

  if bands == 8: img = np.dstack([b0, b1, b2, b3, b4, b5, b6, b7])
  else: img = np.dstack([b0, b1, b2])
  return img

def scaling():
  '''Pipeline function for value scaling. '''

  for i in range(len(p_images)):
    p_images[i] = scale_panchromatic_image(p_images[i])
    m_images[i] = scale_multispectral_image(m_images[i])

## Data Augmentation

In [6]:
def HorizontalFlip():
  '''Performs horizontal flipping. '''

  for i in range(len(polygon_numbers)):
    m_images.append(cv2.flip(m_images[i], 1))
    p_masks.append(cv2.flip(p_masks[i], 1))

def VerticalFlip():
  '''Performs vertical flipping. '''

  for i in range(len(polygon_numbers)):
    m_images.append(cv2.flip(m_images[i], 0))
    p_masks.append(cv2.flip(p_masks[i], 0))

def VHFlip(): 
  '''Performs both horizontal and vertical flipping. '''

  for i in range(len(polygon_numbers)):
    m_images.append(cv2.flip(m_images[i], -1))
    p_masks.append(cv2.flip(p_masks[i], -1))

def Rotation90():
  '''Performs a 90 degrees rotation on the images'''

  for i in range(len(polygon_numbers)):

    # Transpose the image
    image = image.transpose(1,0,2)
    # Flip the image vertically
    image = cv2.flip(image, 1)
    m_images.append(cv2.flip(m_images[i], -1))
    p_masks.append(cv2.flip(p_masks[i], -1))
  
# source: https://www.kaggle.com/safavieh/image-augmentation-using-skimage
import random
import pylab as pl 

def randRange(a, b):
    '''
    a utility function to generate random float values in desired range
    '''
    return pl.rand() * (b - a) + a

def randomCrop(im, msk):
    '''
    croping the image in the center from a random margin from the borders
    '''
    margin = 1/5
    start = [int(randRange(0, im.shape[0] * margin)),
             int(randRange(0, im.shape[1] * margin))]
    end = [int(randRange(im.shape[0] * (1-margin), im.shape[0])), 
           int(randRange(im.shape[1] * (1-margin), im.shape[1]))]
    cropped_image = (im[start[0]:end[0], start[1]:end[1]])
    cropped_mask = (msk[start[0]:end[0], start[1]:end[1]])
    return cropped_image, cropped_mask

def Cropping():
  ''' Applying 10 random croppings to all the images.'''
  
  for i in range(len(polygon_numbers)):
    for j in range(10):
      img, mask = randomCrop(m_images[i], p_masks[i])
      m_images.append(img)
      p_masks.append(mask)

def GaussianBlur():
  '''Applies gaussian blur to the images.'''

  for i in range(len(polygon_numbers)):
    m_images.append(A.GaussianBlur(p=1.0)(image=m_images[i])['image'])
    p_masks.append(p_masks[i])

def data_augmentation():
  '''Performs data augmentation over images and masks. '''

  HorizontalFlip()
  VerticalFlip()
  VHFlip()
  Cropping()
  #GaussianBlur()

In [7]:
def visualize(image, mask, original_image=None, original_mask=None):
    fontsize = 18
    
    if original_image is None and original_mask is None:
        f, ax = plt.subplots(2, 1, figsize=(8, 8))

        ax[0].imshow(image)
        ax[1].imshow(mask)
    else:
        f, ax = plt.subplots(2, 2, figsize=(8, 8))

        ax[0, 0].imshow(original_image)
        ax[0, 0].set_title('Original image', fontsize=fontsize)
        
        ax[1, 0].imshow(original_mask)
        ax[1, 0].set_title('Original mask', fontsize=fontsize)
        
        ax[0, 1].imshow(image)
        ax[0, 1].set_title('Transformed image', fontsize=fontsize)
        
        ax[1, 1].imshow(mask)
        ax[1, 1].set_title('Transformed mask', fontsize=fontsize)

        plt.show()

## Padding

In [8]:
def pad_images(imgs, msks):
  '''Pipeline function for images' padding. '''
  
  images, masks = [], []
  for i in range(len(imgs)):
    images.append(cv2.copyMakeBorder(imgs[i], 99 - imgs[i].shape[0], 0, 80 - imgs[i].shape[1], 0, cv2.BORDER_REFLECT_101))
    masks.append(cv2.copyMakeBorder(msks[i], 99 - msks[i].shape[0], 0, 80 - msks[i].shape[1], 0, cv2.BORDER_REFLECT_101))

  return images, masks

## Preprocessing Pipeline Main Function

In [9]:
def preprocessing_pipeline(method, scale = True): 
  for i in range(len(m_images)):
    m_images[i] = m_images[i].astype(float)
    p_images[i] = p_images[i].astype(float)
  
  if scale: scaling()
  pansharpening(method)
  data_augmentation()
  imgs, masks = pad_images(m_images, p_masks)

  for i in range(len(imgs)):
    masks[i] = masks[i].astype(int)

  return imgs, masks

#m_images, p_masks = preprocessing_pipeline('Simple Mean')

# PyTorch Dataset

In [10]:
#from sklearn.preprocessing import *
from sklearn.decomposition import PCA

def apply_pca(X, transformer = False, components = -1):
    aux = X.copy()
    if transformer:
        X = pd.DataFrame(transformer.fit_transform(X))
        X.columns = aux.columns    
    # Create principal components
    if components == -1:
        pca = PCA()
    else:
        pca = PCA(n_components = components)
        
    X_pca = pca.fit_transform(X)
    # Convert to dataframe
    component_names = [f"PC{i+1}" for i in range(X_pca.shape[1])]
    X_pca = pd.DataFrame(X_pca, columns=component_names)
    # Create loadings
    loadings = pd.DataFrame(
        pca.components_.T,  # transpose the matrix of loadings
        columns=component_names,  # so the columns are the principal components
        index=X.columns,  # and the rows are the original features
    )
    return pca, X_pca, loadings

def plot_variance(pca, width=8, dpi=100):
    # Create figure
    fig, axs = plt.subplots(1, 2, sharey = True)
    n = pca.n_components_
    grid = np.arange(1, n + 1)
    # Explained variance
    evr = pca.explained_variance_ratio_
    axs[0].bar(grid, evr)
    axs[0].set(
        xlabel="Component", title="% Explained Variance")    # ylim = (0.0, 1.0) o sino sharey en plt.subplots
    cv = np.cumsum(evr)
    axs[1].plot(np.r_[0, grid], np.r_[0, cv], "o-")
    axs[1].set(
        xlabel="Component", title="% Cumulative Variance"
    )
    # Set up figure
    fig.set(figwidth=8, dpi=100)
    return axs

def pca_image(img, components = 3, return_pca = False): 
  ''' Performs a PCA and returns a 3-band image with the more significant bands. '''

  image = img.permute(1,2,0).numpy()

  b0 = image[:,:,0]
  b1 = image[:,:,1]
  b2 = image[:,:,2]
  b3 = image[:,:,3]
  b4 = image[:,:,4]
  b5 = image[:,:,5]
  b6 = image[:,:,6]
  b7 = image[:,:,7]

  pca_df = pd.DataFrame({'B0': b0.flatten(), 'B1': b1.flatten(), 'B2': b2.flatten(), 'B3':b3.flatten(), 
              'B4': b4.flatten(), 'B5': b5.flatten(), 'B6': b6.flatten(), 'B7':b7.flatten()})
  
  global_pca, X_pca, loadings = apply_pca(pca_df)

  #X_pca = global_pca.transform(pca_df)
  component_names = [f"PC{i+1}" for i in range(X_pca.shape[1])]
  X_pca = pd.DataFrame(X_pca, columns=component_names)

  sc = MinMaxScaler()
  if components == 3: 
    img = X_pca.loc[:,:'PC3']
    img = pd.DataFrame(sc.fit_transform(img))
    img = img.values.reshape((image.shape[0], image.shape[1], 3))
  
  elif components == 8: 
    img = X_pca
    img = pd.DataFrame(sc.fit_transform(img))
    img = img.values.reshape((image.shape[0], image.shape[1], 8))

  if return_pca == True: return torch.from_numpy(img.astype(float)).permute(2,0,1), global_pca
  else: return torch.from_numpy(img.astype(float)).permute(2,0,1)

def vegetation_indexes(img):
  ''' Returns 3 selected vegetation indexes for the image given.

      %0	Coastal	400	425	450	50	1.24	
      %1	Blue	450	480	510	60	1.24	
      %2	Green	510	545	580	70	1.24	
      %3	Yellow	585	605	625	40	1.24	
      %4	Red	630	660	690	60	1.24	
      %5	Red Edge	705	725	745	40	1.24	
      %6	Near-IR1	770	832.5	895	125	1.24	
      %7	Near-IR2	860	950	1040	180	1.24

      * Normalized Difference Vegetation Index (NDVI) -> (NIR2-Red)/(NIR2 + Red)	UAV/WV-3
      * Green Normalized Difference Vegetation Index  -> (GNDVI)	(NIR2-Green)/(NIR2 + Green)	UAV/WV-3
      * Structure Insensitive Pigment Index (SIPI)	  ->  (NIR1-Blue)/(NIR1 + Red)	WV-3'''

  # Colour Bands
  nir1 = img[6,:,:]
  nir2 = img[7,:,:]
  red = img[4,:,:]
  green = img[2,:,:]
  blue = img[1,:,:]

  #if DEBUG: 
    #print('Valores nulos: ', torch.sum(torch.isnan(img)))

    #print(torch.min(torch.abs(nir2 - red)))
    #print('Numerador - Minimos')
    #print(torch.min(torch.abs(nir2 - green)))
    #print(torch.min(torch.abs(nir1 - blue)))
    #
    #print('Denominador - Minimos')
    #print(torch.min(torch.abs(nir2 + red)))
    #print(torch.min(torch.abs(nir2 + green)))
    #print('Valores no nulos')
    #numpy_array = img.numpy()
    #print(torch.min(torch.abs(nir1 + red)))
    #non_zero_vals = numpy_array[numpy_array != 0]
    #print('Minimo Numpy Array: ', np.min(non_zero_vals))

    #print(np.percentile(torch.abs(nir2 + red).numpy(), 0.0125))
    #print(np.percentile(torch.abs(nir2 + green).numpy(), 0.0125))
    #print(np.percentile(torch.abs(nir1 + red).numpy(), 0.0125))

  # Vegetation Indexes
  # Some images have nan values for "nir2+red", "nir2+green", etc
  numpy_array = (nir2 + red).numpy()
  non_zero_vals = numpy_array[numpy_array != 0]
  non_zero_min = np.min(non_zero_vals)
  ndvi = (nir2 - red) / torch.clamp((nir2 + red), min=non_zero_min)
  ndvi = torch.nan_to_num(ndvi, nan=0.0)

  numpy_array = (nir2 + green).numpy()
  non_zero_vals = numpy_array[numpy_array != 0]
  non_zero_min = np.min(non_zero_vals)
  gndvi = (nir2 - green) / torch.clamp((nir2 + green), min=non_zero_min)
  gndvi = torch.nan_to_num(gndvi, nan=0.0)

  numpy_array = (nir1 + red).numpy()
  non_zero_vals = numpy_array[numpy_array != 0]
  non_zero_min = np.min(non_zero_vals)
  sipi = (nir1 - blue) / torch.clamp((nir1 + red), min=non_zero_min)
  sipi = torch.nan_to_num(sipi, nan=0.0)

  img = torch.stack([ndvi, gndvi, sipi], axis = 0)
  #img = torch.clamp(img, min=-5, max=5)

  if DEBUG: 
    print('Valores nulos: ', torch.sum(torch.isnan(img)).item())
    #print('MIN: ', torch.min(img))
    #print('MAX: ',torch.max(img))
    #print('Percentile 5: ', np.percentile(img.numpy(), .05))
    #print('Percentile 95: ', np.percentile(img.numpy(), .95))

  return img

def extract_image_targets(m):
  ''' Given the mask of a multispectral image, it returns both the instance masks list
      and their respective bounding boxes coordinates. '''

  num_components, masks, stats, centroids = cv2.connectedComponentsWithStats(m.to(torch.uint8).numpy())

  # Find the unique labels in the mask
  labels, counts = torch.unique(torch.from_numpy(masks), return_counts=True)

  # Create an empty list to store the instance masks
  instance_masks = []

  # Iterate over the unique labels and create a separate mask for each one
  for label in labels[1:]:
      instance_mask = torch.where(torch.from_numpy(masks) == label, torch.ones_like(torch.from_numpy(masks)), torch.zeros_like(torch.from_numpy(masks)))
      instance_masks.append(instance_mask)

  # Obtain bounding boxes coordinates
  boxes = torch.zeros([len(instance_masks),4], dtype=torch.float32)
  for i in range(len(instance_masks)):
      x,y,w,h = cv2.boundingRect(instance_masks[i].numpy().astype(np.uint8))
      boxes[i] = torch.tensor([x-1, y-1, x+w, y+h])

  return instance_masks, boxes

def show_masks_with_boxes(instance_masks, boxes): 
  ''' Shows a plot with each object mask, bounded by a rectangle. '''

  n_objects = len(instance_masks)
  plt.figure(figsize=(4*n_objects, 4))
  for i in range(n_objects):
    plt.subplot(1,n_objects, i+1)
    plt.imshow(instance_masks[i])

    # Draw the bounding rectangle
    x1, y1, x2, y2 = boxes[i][0], boxes[i][1], boxes[i][2], boxes[i][3]
    plt.gca().add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='r'))
  plt.show()

In [11]:
class PanchromaticDataset(torch.utils.data.Dataset):
    def __init__(self, images, masks):
      super().__init__()
      self.images = images 
      self.masks = masks 

    def __getitem__(self, i):
      return torch.from_numpy(self.images[i].astype(float)), torch.from_numpy(self.masks[i].astype(float))
        
    def __len__(self):
      return len(self.images)

class MultispectralDataset(torch.utils.data.Dataset):
  def __init__(self, images, masks, approach):
    super().__init__()
    self.images = images 
    self.masks = masks 
    self.approach = approach

  def __getitem__(self, i):
    img, mask = torch.from_numpy(self.images[i].astype(float)).permute(2,0,1), torch.from_numpy(self.masks[i].astype(float))
    
    if self.approach == '': return img, mask
    if self.approach == 'Vegetation Indexes': return vegetation_indexes(torch.nan_to_num(img)), mask
    if self.approach == 'PCA': return pca_image(torch.nan_to_num(img)), mask

  def __len__(self):
      return len(self.images)

if DEBUG: 

  #for p in ['Simple Mean','Brovey','HSV','IHS1','IHS2','IHS3']: 
  #  for d_reduction in ['Vegetation Indexes','PCA']:

  #    p_images, m_images, p_masks, m_masks = [], [], [], []
  #    fit_all_images(polygon_numbers)

  #    m_images2, p_masks2 = preprocessing_pipeline(p, False)
  #    multispectral_dataset = MultispectralDataset(m_images2, p_masks2, d_reduction)

  p_images, m_images, p_masks, m_masks = [], [], [], []
  fit_all_images(polygon_numbers)

  m_images2, p_masks2 = preprocessing_pipeline('IHS1', False)
  multispectral_dataset = MultispectralDataset(m_images2, p_masks2, 'Vegetation Indexes')

  for i in range(1000):
      print('Item: ', i)
      img, mask = multispectral_dataset[i]
      #instance_masks, boxes = extract_image_targets(mask)
      #plt.imshow(img.permute(1,2,0))
      #plt.show()
      #plt.imshow(img.permute(1,2,0))
      #plt.imshow(mask, alpha=.5)
      #plt.show()
      #show_masks_with_boxes(instance_masks, boxes)

  del m_images2, p_masks2, multispectral_dataset, instance_masks, boxes, p_images, m_images, p_masks, m_masks
  gc_collect()

In [12]:
#run = wandb.init(project="Bachelor Thesis", entity="javigallego4")

# loggear las imágenes
#dataloader = torch.utils.data.DataLoader(panchromatic_dataset, batch_size = 1)
#for i, (img, mask) in enumerate(dataloader):
#    wandb.log({"{}".format(i): wandb.Image(img)})

#run = wandb.init(project="Bachelor Thesis", entity="javigallego4")
#dataloader = torch.utils.data.DataLoader(multispectral_dataset, batch_size = 1)
#for i, (img, mask) in tqdm(enumerate(dataloader)):
#  plt.imshow(img.squeeze().permute(1,2,0))
#  wandb.log({"Not Masked: {}".format(i): plt})

#run = wandb.init(project="Bachelor Thesis", entity="javigallego4")

def log_images_wandb(multispectral_dataset, pansharpening, dim_reduction):
  run = wandb.init(project="Bachelor Thesis", entity="javigallego4", group = "Loading Images")
  dataloader = torch.utils.data.DataLoader(multispectral_dataset, batch_size = 1)
  for i, (img, mask) in tqdm(enumerate(dataloader)):
    fig, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (16,8))

    axes[0].imshow(img.squeeze().permute(1,2,0))
    axes[1].imshow(img.squeeze().permute(1,2,0))
    axes[1].imshow(mask.squeeze(), alpha = .5)

    wandb.log({"{}".format(i): fig})

  run.finish()

if LOG_IMAGES: 
  for p in ['PCA', 'Simple Mean', 'Brovey', 'HSV', 'IHS1', 'IHS2', 'IHS3']: 
    for d in ['Vegetation Indexes', 'PCA']:

      p_images, m_images, p_masks, m_masks = [], [], [], []
      fit_all_images(polygon_numbers)

      m_images2, p_masks2 = preprocessing_pipeline(p, False)
      multispectral_dataset = MultispectralDataset(m_images2, p_masks2, d)
      subset = torch.utils.data.Subset(multispectral_dataset, range(261))
      log_images_wandb(subset, p, d)
      clear_output()
      del m_images2, p_masks2, multispectral_dataset, subset, p_images, m_images, p_masks, m_masks
      gc_collect()

# Model

In [13]:
#effb0 = timm.create_model('inception_v4', pretrained = True, num_classes = 0)
#effb0(torch.randn(4,3,99,80)).shape[-1]
#effb0.get_classifier().in_features # 1408 son los effb2.out_channels

# remove the last layer from the backbone. Así obtenemos un feature extractor. OJO Aquí se ve el número de canales para poder usar distintos backbones
#backbone = nn.Sequential(*list(effb0.children())[:-2])
#backbone(torch.randn(1,3,99,80)).shape

In [14]:
effb0 = timm.create_model('efficientnet_b0', pretrained = True)
effb0 = nn.Sequential(*list(effb0.children())[:-2])
effb0(torch.randn(1,3,99,80)).shape

torch.Size([1, 1280, 4, 3])

### Model Implementation

In [15]:
class maskrcnn_model(nn.Module):
    def __init__(self, b):
        super().__init__()

        backbone = timm.create_model(b, pretrained = True)
        out_channels = backbone.get_classifier().in_features
        backbone = nn.Sequential(*list(backbone.children())[:-2])
        backbone.out_channels = out_channels

        print('\n')
        print('=' * 30)
        print(backbone)
        print('=' * 30)
        print('\n')

        anchor_generator = AnchorGenerator(
            sizes=((2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048),),
            aspect_ratios=((1/128, 1/64, 1/32, 1/16, 1/8, 1/4, 1/2, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0, 64, 128),)
        )

        box_roi_pooler = torchvision.ops.MultiScaleRoIAlign(featmap_names=['0'],
                                                        output_size=4,
                                                        sampling_ratio=2)
        
        #rpn_fg_iou_thresh=0.7,
        #rpn_bg_iou_thresh=0.3,
        #rpn_score_thresh=0.0,
        # Box parameters
        #box_score_thresh=0.05,
        #box_fg_iou_thresh=0.75,
        #box_nms_thresh=0.5,
        #box_bg_iou_thresh=0.25,

        #mask_roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        #    featmap_names=['0'], output_size=14, sampling_ratio=2)

        # put the pieces together inside a MaskRCNN model
        #self.model = MaskRCNN(backbone, num_classes=2, rpn_anchor_generator=anchor_generator, box_roi_pool=box_roi_pooler, mask_roi_pool=mask_roi_pooler)
        self.model = MaskRCNN(backbone, num_classes=2, rpn_anchor_generator=anchor_generator, box_roi_pooler=box_roi_pooler, \
                              rpn_fg_iou_thresh=0.7, rpn_bg_iou_thresh=0.3, rpn_score_thresh=0.0, box_score_thresh=0.05, box_nms_thresh=0.5, box_fg_iou_thresh=0.75, box_bg_iou_thresh=0.25,)

        # Change the anchor generator from the predictor
        self.model.roi_heads.box_predictor.anchor_generator = AnchorGenerator(
            sizes=(tuple([i for i in range(1,99)]),),
            aspect_ratios=(tuple([1/i for i in range(2,99)] + [i for i in range(1,99)]),)
        )
        
    def forward(self, inputs, targets):
        #transforms = self.backbone_weights.transforms()
        #inputs = [transforms(d) for d in inputs]
        #targets = [transforms(d) for d in targets]

        x = self.model(inputs, targets)
        return x
      
    def predict(self, x):
        #transforms = self.backbone_weights.transforms()
        #x = [transforms(d) for d in x]
        x = self.model(x)
        return x

# Training Pipeline

### Helper Functions

In [16]:
class EarlyStopping():
    """
    Early stopping to stop the training when the loss does not improve after
    certain epochs.
    """
    def __init__(self, patience=5, min_delta=0):
        """
        :param patience: how many epochs to wait before stopping when loss is
               not improving
        :param min_delta: minimum difference between new loss and old loss for
               new loss to be considered as an improvement
        """
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss == None:
            self.best_loss = val_loss
            
        elif self.best_loss - val_loss > self.min_delta:
            self.best_loss = val_loss
            # reset counter if validation loss improves
            self.counter = 0
            
        elif self.best_loss - val_loss <= self.min_delta:
            self.counter += 1
            print(f"INFO: Early stopping counter {self.counter} of {self.patience}")
            if self.counter >= self.patience:
                print('INFO: Early stopping')
                self.early_stop = True

In [17]:
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

def train_one_epoch(train_loader, model, optimizer, scaler):
    # Metric
    metric = MeanAveragePrecision(iou_type = 'bbox')

    # Track losses
    loss_epoch = 0
    mAP_epoch, mAP50_epoch, mAP75_epoch = 0, 0, 0

    # Loop over minibatches
    for imgs, masks in tqdm(train_loader):
        model.train() # Train mode

        # Create MaskRCNN inputs and targets
        inputs = [imgs[i].to(device, dtype = torch.float32, non_blocking = True) for i in range(imgs.shape[0])]
        targets = []

        for i in range(imgs.shape[0]):
          instance_masks, boxes = extract_image_targets(masks[i].squeeze())
          target = {}
          target["boxes"] = boxes.to(device, non_blocking = True)
          target["labels"] = torch.ones(len(instance_masks), dtype=torch.int64).to(device, non_blocking = True)   
          target["masks"] = torch.stack(instance_masks).to(torch.uint8).to(device, non_blocking = True)
          targets.append(target)

        # Zero gradients
        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        with torch.cuda.amp.autocast():
          outputs = model(inputs, targets)

        losses = sum(loss for loss in outputs.values())
        #losses.backward()
        #optimizer.step()

        # Scales the loss, and calls backward() to create scaled gradients
        scaler.scale(losses).backward()

        # Unscales gradients and calls or skips optimizer.step()
        scaler.step(optimizer)

        # Updates the scale for next iteration
        scaler.update()
            
        # Backprop
        #loss.backward()

        # Update parameters
        #optimizer.step()

        # Track losses
        loss_epoch += losses.detach().item()
        #loss_classifier += outputs['loss_classifier'].detach().item()
        #loss_box_reg += outputs['loss_box_reg'].detach().item()
        #loss_mask += outputs['loss_mask'].detach().item()
        #loss_objectness += outputs['loss_objectness'].detach().item()
        #loss_rpn_box_reg += outputs['loss_rpn_box_reg'].detach().item()

        # Track metric
        # Don't update weights
        with torch.no_grad():
          model.eval()
          preds = model.predict(inputs)

          if DEBUG: 
            for i in range(4):
              p = preds[i]
              t = targets[i]
              #print(t['masks'].shape)
              for j in range(p['masks'].shape[0]): 
                print('Valores nulos', torch.sum(torch.isnan(p['masks'][j])).item())
                print('Mediana', torch.median(p['masks'][j]))
                plt.imshow(p['masks'][j].permute(1,2,0).cpu().numpy())
                plt.show()

              print('\n\n\nTargets')
              for j in range(t['masks'].shape[0]):
                plt.imshow(t['masks'][j].cpu().numpy())
                plt.show()

          for pred in preds: 
            pred.pop('masks', None)

          for target in targets:
            target.pop('masks', None)
  
          metric.update(preds, targets)
          m = metric.compute()
          mAP_epoch += m['map']
          mAP50_epoch += m['map_50']
          mAP75_epoch += m['map_75']
        
        del inputs, targets, preds, outputs
        gc_collect()
        
    return loss_epoch/len(train_loader), mAP_epoch/len(train_loader), mAP50_epoch/len(train_loader), mAP75_epoch/len(train_loader), model

if DEBUG: 
  gc_collect()
  model = maskrcnn_model('efficientnet_b0').to(device)

  p_images, m_images, p_masks, m_masks = [], [], [], []
  fit_all_images(polygon_numbers)

  m_images2, p_masks2 = preprocessing_pipeline('HSV', False)
  multispectral_dataset = MultispectralDataset(m_images2, p_masks2, 'PCA')

  dataloader = torch.utils.data.DataLoader(multispectral_dataset, batch_size = 4, num_workers = os.cpu_count(), pin_memory=True)
  optimizer = optim.AdamW(model.parameters())
  scaler = torch.cuda.amp.GradScaler()
  loss, mAP, mAP50, mAP75, model = train_one_epoch(dataloader, model, optimizer, scaler)

In [18]:
def validate_one_epoch(validation_loader, model):
    metric = MeanAveragePrecision(iou_type = 'bbox')
    loss_epoch = 0
    mAP_epoch, mAP50_epoch, mAP75_epoch = 0, 0, 0
    
    # Don't update weights
    with torch.no_grad():

      # Loop over minibatches
      for imgs, masks in tqdm(validation_loader):
          model.train()

          # Create MaskRCNN inputs and targets
          inputs = [imgs[i].to(device, dtype = torch.float32) for i in range(imgs.shape[0])]
          targets = []

          for i in range(imgs.shape[0]):
            instance_masks, boxes = extract_image_targets(masks[i].squeeze())
            target = {}
            target["boxes"] = boxes.to(device)
            target["labels"] = torch.ones(len(instance_masks), dtype=torch.int64).to(device)   
            target["masks"] = torch.stack(instance_masks).to(torch.uint8).to(device)
            targets.append(target)

          # Make predictions and obtain losses
          with torch.cuda.amp.autocast():
            outputs = model(inputs, targets)

          losses = sum(loss for loss in outputs.values())

          # Track loss
          loss_epoch += losses.detach().item()

          # Track metric
          model.eval()
          preds = model.predict(inputs)

          for pred in preds: 
            pred.pop('masks', None)
          
          for target in targets:
            target.pop('masks', None)
 
          metric.update(preds, targets)
          m = metric.compute()
          mAP_epoch += m['map']
          mAP50_epoch += m['map_50']
          mAP75_epoch += m['map_75']

          del inputs, targets, preds
          gc_collect()
            
    return loss_epoch/len(validation_loader), mAP_epoch/len(validation_loader), mAP50_epoch/len(validation_loader), mAP75_epoch/len(validation_loader)
    #return loss_epoch/len(validation_loader), pfbeta_epoch/len(validation_loader), threshold

### Main Function

In [19]:
def train_model(verbose=True):
    torch.manual_seed(42)
    # Init W&B 
    run = wandb.init(project="Bachelor Thesis", entity="javigallego4", group = 'Tuning')

    # Preprocessing Pipeline (No initial scaling)
    #p_images, m_images, p_masks, m_masks = [], [], [], []
    #fit_all_images(polygon_numbers)
    m_images2, p_masks2 = preprocessing_pipeline(wandb.config.pansharpening, False)

    print('Preprocessing done')

    # Datasets and train-test-split
    multispectral_dataset = MultispectralDataset(m_images2, p_masks2, wandb.config.dimensionality_reduction)
    n = len(multispectral_dataset)
    train_data, validation_data = random_split(multispectral_dataset, [int(n*0.8), int(n*0.2)+1])
    del m_images2, p_masks2
    gc_collect()

    print('Dataset created')

    # Model
    model = maskrcnn_model(wandb.config.backbone).to(device)
    print('Model created')
    
    # Freezing initial layers
    #NUM_FROZEN_LAYERS = int(len(list(model.named_parameters())) * wandb.config.freezing) # how many layers you want to freeze
    #for name, param in list(model.named_parameters())[0:NUM_FROZEN_LAYERS]:
    #    param.requires_grad = False

    # Construct an optimizer and scheduler
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params, lr = wandb.config.lr, weight_decay = wandb.config.weight_decay, amsgrad = wandb.config.amsgrad) if wandb.config.optimizer == 'AdamW' \
                else optim.SGD(params, lr=wandb.config.lr)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=2, threshold = wandb.config.threshold, verbose = True)

    # Gradient scaling helps prevent gradients with small magnitudes from flushing to zero (“underflowing”) when training with mixed precision.
    scaler = torch.cuda.amp.GradScaler()
    
    #for i in range(wandb.config.folds): 
    #   train_idx = train[train.VAL_FOLD != i].index
    #    val_idx = train[train.VAL_FOLD == i].index
    #batch = wandb.config.batch_size if wandb.config.backbone == 'ResNet-50-FPN' else int(wandb.config.batch_size / 2)     
    batch = 1
    trainloader = torch.utils.data.DataLoader(train_data, batch_size = batch, num_workers = 4, shuffle = True, pin_memory=True)
    validationloader = torch.utils.data.DataLoader(validation_data, batch_size = batch, num_workers = 4, pin_memory=True)
        
        #print('====== Fold: {} ======='.format(i))
    
    best_mAP = 0
    early_stopping = EarlyStopping(5, 0.005)

    # Loop over epochs
    for epoch in range(wandb.config.epochs):
            
          # Train
          train_loss, train_mAP, train_mAP50, train_mAP75, model = train_one_epoch(trainloader, model, optimizer, scaler)
          
          # Evaluate
          val_loss, val_mAP, val_mAP50, val_mAP75 = validate_one_epoch(validationloader, model)

          # Apply scheduler
          scheduler.step(val_loss)
          
          # Log metrics
          wandb.log({
              'epoch': epoch,
              'train_loss': train_loss,
              'val_loss': val_loss,
              'train_mAP': train_mAP,
              'train_mAP50': train_mAP50, 
              'train_mAP75': train_mAP75, 
              'val_mAP': val_mAP,
              'val_mAP50': val_mAP50,
              'val_mAP75': val_mAP75 
          })

          # Print loss
          if verbose:
              if (epoch+1)%1==0:
                  print(f'\nEpoch {epoch+1}/{wandb.config.epochs}')
                  print(f'loss {train_loss:.5f}, val_loss {val_loss:.5f}')
                  print(f'mAP {train_mAP:.5f}, mAP50 {train_mAP50:.5f}, mAP75 {train_mAP75:.5f}, val_mAP {val_mAP:.10f}, val_mAP50 {val_mAP50:.5f}, val_mAP75 {val_mAP75:.5f}')
  
          # Early Stopping
          early_stopping(val_loss)
          if early_stopping.early_stop:
              break
          else:
              if val_mAP > best_mAP: 
                  best_mAP = val_mAP
                  best_model_state_dic = model.state_dict()

          # Every 10 epochs, save the best mAP and best model as W&B Artifact  
          if (epoch+1)%10 == 0: 
            wandb.log({'best_mAP':best_mAP})
            PATH = "{}.pt".format(wandb.run.name)
            torch.save(best_model_state_dic, PATH)

            artifact = wandb.Artifact(name='{}'.format(run.name), type='model')
            artifact.add_file('{}.pt'.format(run.name))
            run.log_artifact(artifact)

          print('\n')

    #cv_val_metric.append(best_pf1)
    wandb.log({'best_mAP':best_mAP})
    #wandb.log({'cv_val_metric': np.mean(cv_val_metric)})
  
    # Save the best model as W&B Artifact  
    PATH = "{}.pt".format(wandb.run.name)
    torch.save(best_model_state_dic, PATH)

    artifact = wandb.Artifact(name='{}'.format(run.name), type='model')
    artifact.add_file('{}.pt'.format(run.name))
    run.log_artifact(artifact)
    run.finish()

### W&B Sweeps

In [20]:
%env SWEEP_ID=re15k1n2
#os.environ.__delitem__('SWEEP_ID')

env: SWEEP_ID=re15k1n2


In [ ]:
gc_collect()
sweep_id = os.environ.get('SWEEP_ID')
print('wandb sweep ', sweep_id)

if sweep_id is None:
    # Define the sweep configuration
    sweep_id = wandb.sweep(sweep={
            'method': 'bayes',
            'name': 'PCA Reduction + MaskRCNN',
            'metric': {'goal': 'maximize', 'name': 'val_mAP'},
            'parameters':
                {   
                    # Preprocessing and Dataset Config
                    'pansharpening': {'values': ['Simple Mean', 'Brovey', 'HSV', 'IHS1', 'IHS2', 'IHS3']},
                    'dimensionality_reduction': {'values': ['PCA']},

                    # Model config
                    #'backbone': {'values': ['ResNet-50-FPN', 'ResNet-50-FPN V2', 'MobileNet V2']},
                    'backbone': {'values': ['efficientnet_b{}'.format(i) for i in range(5)] + ['mobilenetv2_100','resnext101_32x8d','resnet18','resnet50','resnet101']},

                    # Training config
                    'batch_size' : {'values': [8]}, 
                    'epochs': {'values': [60]},
                    'freezing': {'values': [0.25, 0.5, 0.75]}, 
                    'image_size': {'values': [(99,80)]},
                    'dataset_split': {'values': [(0.8, 0.2)]},
                    
                    # Optimizer configuration.  
                    'optimizer': {'values': ['AdamW']},
                    'lr': {'distribution': 'uniform', "min": 1e-04, "max": 1e-02}, 
                    'weight_decay': {'distribution': 'uniform', "min": 1e-03, "max": 1e-01},
                    'amsgrad': {'values': [True, False]},
                 
                    # Lr Scheduler
                    'scheduler': {'values': ['ReduceLROnPlateau']},
                    'threshold': {'values': [0.005]},

                    # Metric 
                    'mAP_type': {'values': ['bbox']},
                    
                }
        }, project="Bachelor Thesis")

    print('Generated sweep id', sweep_id)

else:
    """
    Agent run. Use sweep_id generated above to produce (semi)-random hyperparameters run.config
    """
    wandb.agent(sweep_id, function=train_model, entity="javigallego4", project="Bachelor Thesis", count = 1)

wandb sweep  re15k1n2


wandb: Agent Starting Run: w28xdzrj with config:
wandb: 	amsgrad: True
wandb: 	backbone: resnet18
wandb: 	batch_size: 8
wandb: 	dataset_split: [0.8, 0.2]
wandb: 	dimensionality_reduction: PCA
wandb: 	epochs: 60
wandb: 	freezing: 0.25
wandb: 	image_size: [99, 80]
wandb: 	lr: 0.006107473107209394
wandb: 	mAP_type: bbox
wandb: 	optimizer: AdamW
wandb: 	pansharpening: IHS3
wandb: 	scheduler: ReduceLROnPlateau
wandb: 	threshold: 0.005
wandb: 	weight_decay: 0.028829317437956515
ERROR:wandb.jupyter:Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING Ignored wandb.init() arg project when running a sweep.
wandb: WARNING Ignored wandb.init() arg entity when running a sweep.


Preprocessing done
Dataset created


Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to /root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth




Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act2): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, a

 12%|█▏        | 351/2923 [14:52<2:07:09,  2.97s/it]